# Melhora iterativa: fluxo máximo e emparelhamento bipartido — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As verificações, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- **validar** um fluxo (capacidade e conservação) e achar o **corte mínimo** por força bruta em redes pequenas;
- implementar **Ford-Fulkerson com BFS** (Edmonds-Karp) em C, com a rede residual em listas de adjacência;
- extrair o **corte mínimo** da última rede residual e usá-lo como certificado de que o fluxo é máximo;
- medir o efeito da **escolha do caminho aumentador** no número de aumentos;
- implementar o **emparelhamento bipartido máximo** por caminhos alternantes em C;
- **modelar** um problema como fluxo (desafio: eliminação no campeonato).

Os vértices da rede condutora (Cormen et al., Figura 26.1) são $s, v_1, v_2, v_3, v_4, t$; nos
programas eles viram os números $0, 1, 2, 3, 4, 5$. Em todas as redes deste tutorial, **a origem é
o vértice $0$ e o sorvedouro é o vértice $n - 1$**.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess, math, random, itertools
from collections import deque

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True,
                       errors='replace')
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor inteiro de 'nome=valor' da saída."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            return int(tok.split('=', 1)[1])
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

# Rede condutora: Cormen et al. (2009), §26.1, Figura 26.1. Arcos (u, v, capacidade).
NOMES = ['s', 'v1', 'v2', 'v3', 'v4', 't']
REDE = [(0, 1, 16), (0, 2, 13), (1, 3, 12), (2, 1, 4), (2, 4, 14),
        (3, 2, 9), (3, 5, 20), (4, 3, 7), (4, 5, 4)]
# Rede em que o guloso trava (Kleinberg e Tardos, §7.1): s=0, u=1, v=2, t=3.
REDE_KT = [(0, 1, 20), (0, 2, 10), (1, 2, 30), (1, 3, 10), (2, 3, 20)]

def entrada_rede(n, arcos):
    """Formato lido pelos programas em C: 'n m' e depois m linhas 'u v c' (origem 0, sorvedouro n-1)."""
    linhas = [f'{n} {len(arcos)}'] + [f'{u} {v} {c}' for u, v, c in arcos]
    return '\n'.join(linhas) + '\n'

def rede_aleatoria(n, m, rng, cmax=10):
    """m arcos u -> v (u != v, sem arcos entrando na origem nem saindo do sorvedouro)."""
    arcos = []
    while len(arcos) < m:
        u, v = rng.randrange(n - 1), rng.randrange(1, n)
        if u != v:
            arcos.append((u, v, rng.randint(1, cmax)))
    return arcos

# Posições dos vértices no desenho (as mesmas dos slides).
POS = {0: (0, 0), 1: (1.7, 1), 2: (1.7, -1), 3: (3.6, 1), 4: (3.6, -1), 5: (5.3, 0)}

In [ ]:
# Desenho de redes pequenas: rótulo "f/c" em cada arco, arcos destacados em laranja.
import matplotlib.pyplot as plt

def desenha(n, arcos, fluxo=None, pos=None, nomes=None, destaque=(), S=None, titulo='', ax=None):
    """fluxo: lista com o fluxo de cada arco (mesma ordem de arcos); S: vértices sombreados."""
    if pos is None:
        pos = {v: (math.cos(2 * math.pi * v / n), math.sin(2 * math.pi * v / n)) for v in range(n)}
    nomes = nomes or [str(v) for v in range(n)]
    destaque = set(destaque)
    pares = {(u, v) for u, v, _ in arcos}
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 2.8))
    for i, (u, v, c) in enumerate(arcos):
        (x1, y1), (x2, y2) = pos[u], pos[v]
        rad = 0.25 if (v, u) in pares else 0.0
        dentro = (u, v) in destaque
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1), zorder=1,
                    arrowprops=dict(arrowstyle='-|>', shrinkA=11, shrinkB=11,
                                    connectionstyle=f'arc3,rad={rad}',
                                    color='darkorange' if dentro else 'gray',
                                    lw=2.8 if dentro else 1))
        cx = (x1 + x2) / 2 + rad * (y2 - y1) / 2
        cy = (y1 + y2) / 2 - rad * (x2 - x1) / 2
        rot = str(c) if fluxo is None else f'{fluxo[i]}/{c}'
        ax.text(cx, cy, rot, fontsize=8, ha='center', va='center', zorder=2,
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none'))
    for v, (x, y) in pos.items():
        cor = 'navajowhite' if S and v in S else 'lightsteelblue'
        ax.scatter(x, y, s=380, zorder=3, color=cor, edgecolor='steelblue')
        ax.text(x, y, nomes[v], ha='center', va='center', fontsize=9, zorder=4)
    ax.set_title(titulo); ax.axis('off'); ax.set_aspect('equal'); ax.margins(0.12)
    return ax

desenha(6, REDE, pos=POS, nomes=NOMES, titulo='A rede condutora (Cormen et al., Figura 26.1)')
plt.show()

## 1. Fluxos e cortes

Um **fluxo** atribui a cada arco $(u,v)$ um valor $f(u,v)$ com $0 \le f(u,v) \le c(u,v)$
(**capacidade**) e, em todo vértice diferente de $s$ e $t$, o que entra é igual ao que sai
(**conservação**). O **valor** $|f|$ é o saldo que sai de $s$.

Um **corte** $(S, T)$ separa os vértices com $s \in S$ e $t \in T$. Sua capacidade $c(S,T)$ soma
só os arcos que vão de $S$ para $T$. Todo fluxo passa inteiro por todo corte, logo
$|f| \le c(S,T)$ para **qualquer** par fluxo/corte. O teorema de Ford e Fulkerson garante que o
valor do fluxo máximo é igual à capacidade do corte mínimo.

A célula abaixo valida o fluxo de valor 16 dos slides. A função `valida_fluxo` será usada para
testar os seus programas.

In [ ]:
# Exemplo: validar um fluxo e calcular seu valor.

def valida_fluxo(n, arcos, fluxo, s=0, t=None):
    """Devolve o valor do fluxo se ele respeita capacidade e conservação; senão levanta AssertionError."""
    t = n - 1 if t is None else t
    saldo = [0] * n                                   # saldo[v] = sai - entra
    for (u, v, c), f in zip(arcos, fluxo):
        assert 0 <= f <= c, f'arco {u}->{v}: fluxo {f} fora de [0, {c}]'
        saldo[u] += f
        saldo[v] -= f
    for v in range(n):
        if v not in (s, t):
            assert saldo[v] == 0, f'vértice {v}: sai - entra = {saldo[v]}, deveria ser 0'
    return saldo[s]

def capacidade_corte(arcos, S):
    """Soma das capacidades dos arcos que saem de S."""
    return sum(c for u, v, c in arcos if u in S and v not in S)

F16 = [12, 4, 12, 0, 4, 0, 12, 0, 4]                  # o fluxo de valor 16 dos slides
print('valor do fluxo:', valida_fluxo(6, REDE, F16))
print('capacidade do corte S = {s, v1}:', capacidade_corte(REDE, {0, 1}))

desenha(6, REDE, F16, POS, NOMES, destaque=[(1, 3), (4, 5)],
        titulo='Fluxo de valor 16 (em laranja, os arcos cheios)')
plt.show()

In [ ]:
# Exercício 1: corte mínimo por força bruta
# TODO: enumere todos os conjuntos S com s = 0 em S e t = n-1 fora de S (são 2^(n-2)) e
# devolva (capacidade, S) do corte de menor capacidade. S deve ser um set de vértices.

def corte_minimo_forca_bruta(n, arcos):
    # TODO: implemente aqui
    pass

# Teste automático
cap, S = corte_minimo_forca_bruta(6, REDE)
assert cap == 23 and capacidade_corte(REDE, S) == 23 and 0 in S and 5 not in S, 'Verifique sua implementação'
assert corte_minimo_forca_bruta(4, REDE_KT)[0] == 30, 'Verifique sua implementação'
print('corte mínimo da rede condutora:', cap, sorted(NOMES[v] for v in S))
print('Exercício 1 ok!')

A força bruta custa $\Theta(2^{n} m)$: com $n = 40$ já são mais de $10^{11}$ cortes. Ela serve
para **testar** os algoritmos eficientes em redes pequenas, que é como vamos usá-la.

## 2. Ford-Fulkerson com BFS (Edmonds-Karp) em C

Nos slides, a rede residual era uma matriz $n \times n$ e cada BFS custava $\Theta(n^2)$. Com
**listas de adjacência**, a BFS custa $O(n + m)$. O truque é guardar cada arco original $i$ como
**dois** arcos residuais consecutivos:

- o arco $a = 2i$, de **avanço** $u \to v$, começa com capacidade residual $c$;
- o arco $a \oplus 1 = 2i + 1$, de **retorno** $v \to u$, começa com capacidade residual $0$.

Assim, `a ^ 1` é sempre o arco "irmão" de `a`: aumentar $b$ unidades em `a` é fazer
`cap[a] -= b; cap[a ^ 1] += b`. No fim, o fluxo no arco original $i$ é `cap[2*i + 1]`.

### Exercício 2: Edmonds-Karp com listas de adjacência

Complete `fluxo_maximo(s, t, &aumentos)`. Em cada rodada:

1. faça uma BFS a partir de `s` usando só arcos residuais com `cap[a] > 0`, guardando em
   `arcoPai[v]` o arco pelo qual a BFS chegou a `v` (`-1` se `v` não foi alcançado);
2. se `t` não foi alcançado, devolva o fluxo total;
3. senão, ache o gargalo $b$ subindo de `t` até `s` por `arcoPai[]` (a origem do arco `a` é
   `dest[a ^ 1]`), aumente $b$ em cada arco do caminho e some $b$ ao total.

Conte em `*aumentos` o número de caminhos usados. O programa lê a rede da entrada e imprime
`fluxo=… aumentos=…` seguido do fluxo em cada arco original; a célula de teste confere o
resultado com `valida_fluxo` e com a força bruta do Exercício 1 em 300 redes aleatórias.

In [ ]:
%%writefile src/fluxo.c
/* Exercicio 2: fluxo maximo por Ford-Fulkerson com BFS (Edmonds-Karp): O(n m^2).
   Entrada: "n m" e m linhas "u v c"; origem 0, sorvedouro n-1.
   TODO: complete fluxo_maximo(). */
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>

/* rede residual em listas de adjacencia: o arco residual a vai para dest[a], tem capacidade
   residual cap[a]; o irmao (retorno) de a e a ^ 1, e a origem de a e dest[a ^ 1] */
int n, m;
int *primeiro, *prox, *dest, *cap;
int *arcoPai, *fila;

void insere_arco( int u, int v, int c) {          /* cria os arcos 2i (u->v) e 2i+1 (v->u) */
   static int a = 0;
   dest[a] = v;  cap[a] = c;  prox[a] = primeiro[u];  primeiro[u] = a;  ++a;
   dest[a] = u;  cap[a] = 0;  prox[a] = primeiro[v];  primeiro[v] = a;  ++a;
}

/* devolve o valor do fluxo maximo de s a t; *aumentos recebe o numero de caminhos usados */
int fluxo_maximo( int s, int t, int *aumentos) {
   /* TODO */
   return 0;
}

/* ---------- leitura e saida (nao altere) ---------- */
int main( void) {
   if (scanf( "%d %d", &n, &m) != 2) return 1;
   primeiro = malloc( n * sizeof (int));
   prox = malloc( 2 * m * sizeof (int));  dest = malloc( 2 * m * sizeof (int));
   cap = malloc( 2 * m * sizeof (int));
   arcoPai = malloc( n * sizeof (int));  fila = malloc( n * sizeof (int));
   for (int v = 0; v < n; ++v) primeiro[v] = -1;
   for (int i = 0; i < m; ++i) {
      int u, v, c;
      if (scanf( "%d %d %d", &u, &v, &c) != 3) return 1;
      insere_arco( u, v, c);
   }
   int aumentos = 0;
   int valor = fluxo_maximo( 0, n - 1, &aumentos);
   printf( "fluxo=%d aumentos=%d\n", valor, aumentos);
   for (int i = 0; i < m; ++i) printf( "%d\n", cap[2*i + 1]);   /* fluxo no arco i */
   return 0;
}

In [ ]:
# Teste automático do Exercício 2

def fluxo_c(n, arcos, mostrar=False):
    """Roda src/fluxo.c; devolve (valor, aumentos, lista com o fluxo em cada arco)."""
    saida = rodar('fluxo.c', entrada=entrada_rede(n, arcos), mostrar=mostrar)
    linhas = saida.split('\n')
    return campo(linhas[0], 'fluxo'), campo(linhas[0], 'aumentos'), [int(x) for x in linhas[1:1 + len(arcos)]]

valor, aum, f = fluxo_c(6, REDE, mostrar=True)
assert valor == 23 and aum == 3, 'Verifique sua implementação (rede condutora: 23 em 3 aumentos)'
assert valida_fluxo(6, REDE, f) == 23, 'O fluxo impresso não é válido'
assert fluxo_c(4, REDE_KT)[0] == 30, 'Verifique sua implementação (rede de Kleinberg e Tardos: 30)'

rng = random.Random(2026)
for _ in range(300):
    n = rng.randint(2, 8)
    arcos = rede_aleatoria(n, rng.randint(1, 16), rng)
    valor, _, f = fluxo_c(n, arcos)
    assert valida_fluxo(n, arcos, f) == valor, f'fluxo inválido em {arcos}'
    assert valor == corte_minimo_forca_bruta(n, arcos)[0], f'fluxo não é máximo em {arcos}'
print('Exercício 2 ok!')

In [ ]:
# Visualização: o fluxo máximo da rede condutora, com os arcos cheios em laranja.
valor, aum, f = fluxo_c(6, REDE)
cheios = [(u, v) for (u, v, c), x in zip(REDE, f) if x == c]
desenha(6, REDE, f, POS, NOMES, destaque=cheios, titulo=f'Fluxo máximo: {valor} em {aum} aumentos')
plt.show()

## 3. O corte mínimo como certificado

Quando Ford-Fulkerson para, o conjunto $S$ dos vértices **alcançáveis a partir de $s$ na rede
residual** forma um corte mínimo: todo arco de $S$ para $T$ está cheio e todo arco de $T$ para
$S$ está vazio. Logo $c(S,T) = |f|$, e esse corte **prova** que o fluxo é máximo, sem precisar
confiar no programa que o calculou.

A célula abaixo traz uma referência em Python (a mesma ideia do Exercício 2, com matriz) que
vamos usar nas seções seguintes.

In [ ]:
# Referência: Edmonds-Karp em Python com matriz residual (devolve valor, aumentos e fluxo por arco).

def fluxo_max_ref(n, arcos, s=0, t=None):
    t = n - 1 if t is None else t
    r = [[0] * n for _ in range(n)]
    for u, v, c in arcos:
        r[u][v] += c
    total = aumentos = 0
    while True:
        pai = [-1] * n
        pai[s] = s
        fila = deque([s])
        while fila and pai[t] == -1:
            u = fila.popleft()
            for v in range(n):
                if pai[v] == -1 and r[u][v] > 0:
                    pai[v] = u
                    fila.append(v)
        if pai[t] == -1:
            break
        b, v = math.inf, t
        while v != s:
            b, v = min(b, r[pai[v]][v]), pai[v]
        v = t
        while v != s:
            r[pai[v]][v] -= b
            r[v][pai[v]] += b
            v = pai[v]
        total += b
        aumentos += 1
    # fluxo líquido de cada par (u, v), repartido entre os arcos originais paralelos
    capac = [[0] * n for _ in range(n)]
    for u, v, c in arcos:
        capac[u][v] += c
    liquido = {(u, v): max(0, capac[u][v] - r[u][v]) for u, v, _ in arcos}
    fluxo = []
    for u, v, c in arcos:
        x = min(c, liquido[(u, v)])
        liquido[(u, v)] -= x
        fluxo.append(x)
    return total, aumentos, fluxo

print(fluxo_max_ref(6, REDE))

In [ ]:
# Exercício 3: corte mínimo a partir do fluxo máximo
# TODO: monte a rede residual do fluxo dado (avanço u->v se f < c; retorno v->u se f > 0),
# faça uma busca a partir de s = 0 e devolva o conjunto S dos vértices alcançados.

def corte_do_fluxo(n, arcos, fluxo):
    # TODO: implemente aqui
    pass

# Teste automático
F23 = [12, 11, 12, 0, 11, 0, 19, 7, 4]               # o fluxo máximo dos slides
S = corte_do_fluxo(6, REDE, F23)
assert S == {0, 1, 2, 4}, 'Verifique sua implementação (esperado S = {s, v1, v2, v4})'
rng = random.Random(7)
for _ in range(300):
    n = rng.randint(2, 9)
    arcos = rede_aleatoria(n, rng.randint(1, 18), rng)
    valor, _, f = fluxo_max_ref(n, arcos)
    S = corte_do_fluxo(n, arcos, f)
    assert 0 in S and n - 1 not in S and capacidade_corte(arcos, S) == valor, f'corte errado em {arcos}'
print('Exercício 3 ok!')

In [ ]:
# Visualização: o certificado na rede condutora (S sombreado, arcos de S para T em laranja).
S = corte_do_fluxo(6, REDE, F23)
cruzam = [(u, v) for u, v, c in REDE if u in S and v not in S]
desenha(6, REDE, F23, POS, NOMES, destaque=cruzam, S=S,
        titulo=f'Corte mínimo: c(S,T) = {capacidade_corte(REDE, S)} = |f|')
plt.show()

## 4. A escolha do caminho importa

Na rede abaixo (Cormen et al., Figura 26.7), as capacidades valem $C$, exceto o arco central
$u \to v$, que vale $1$. Um Ford-Fulkerson **azarado** alterna os caminhos
$s \to u \to v \to t$ e $s \to v \to u \to t$ (este pelo retorno $v \to u$), cada um com gargalo
$1$, e faz $2C$ aumentos. A BFS de Edmonds-Karp escolhe sempre um caminho com menos arcos e
termina em 2 aumentos, qualquer que seja $C$.

A célula a seguir simula o Ford-Fulkerson azarado e compara com o seu programa do Exercício 2.

In [ ]:
# Visualização: número de aumentos em função de C, azarado contra Edmonds-Karp.

def rede_ruim(C):
    return [(0, 1, C), (0, 2, C), (1, 2, 1), (1, 3, C), (2, 3, C)]   # s=0, u=1, v=2, t=3

def aumentos_azarado(C):
    """Simula Ford-Fulkerson escolhendo sempre um caminho pelo arco central (gargalo 1)."""
    r = {(0, 1): C, (0, 2): C, (1, 2): 1, (2, 1): 0, (1, 3): C, (2, 3): C}
    aum = 0
    while True:
        if r[(0, 1)] and r[(1, 2)] and r[(2, 3)]:            # s -> u -> v -> t
            for a in ((0, 1), (1, 2), (2, 3)): r[a] -= 1
            r[(2, 1)] += 1
        elif r[(0, 2)] and r[(2, 1)] and r[(1, 3)]:          # s -> v -> u -> t (retorno)
            for a in ((0, 2), (2, 1), (1, 3)): r[a] -= 1
            r[(1, 2)] += 1
        else:
            break
        aum += 1
    return aum

Cs = [1, 10, 100, 1000, 10000, 100000]
azar = [aumentos_azarado(C) for C in Cs]
ek = [fluxo_c(4, rede_ruim(C))[1] for C in Cs]
for C, a, e in zip(Cs, azar, ek):
    print(f'C = {C:>6}: azarado {a:>6} aumentos, Edmonds-Karp {e}')

plt.figure(figsize=(6, 3.4))
plt.plot(Cs, azar, 'o-', label='Ford-Fulkerson azarado: 2C')
plt.plot(Cs, ek, 's-', label='Edmonds-Karp (BFS)')
plt.xscale('log'); plt.yscale('log'); plt.xlabel('C'); plt.ylabel('aumentos'); plt.legend()
plt.title('O custo depende das capacidades se o caminho for mal escolhido')
plt.tight_layout(); plt.show()

O Ford-Fulkerson azarado precisaria de $2 \times 10^6$ aumentos para $C = 10^6$, embora a rede
tenha só 4 vértices. Com BFS, o número de aumentos é $O(nm)$, independente das capacidades
(Edmonds e Karp, 1972).

## 5. Emparelhamento bipartido

Num grafo bipartido (alunos $L$, projetos $R$), um **emparelhamento** é um conjunto de arestas
sem ponta em comum. Ligando $s$ a cada aluno, cada projeto a $t$ e pondo capacidade 1 em tudo,
o emparelhamento máximo é o fluxo máximo. Na prática, nem é preciso montar a rede: um caminho
aumentador nela é um **caminho alternante** no grafo original, que sai de um aluno livre, usa
uma aresta livre até um projeto, volta pela aresta emparelhada desse projeto até o aluno dono
dele, e assim por diante, até achar um projeto livre.

A versão por busca em profundidade é curta (algoritmo de Kuhn): para encaixar o aluno `u`, tente
cada projeto `v` vizinho; se `v` está livre, ou se o dono atual de `v` consegue, recursivamente,
ser encaixado em **outro** projeto, então `v` fica com `u`.

### Exercício 4: emparelhamento máximo por caminhos alternantes

Complete `aumenta(u)` e `emparelhamento_maximo()`:

- `aumenta(u)` procura um caminho alternante a partir do aluno `u`. Para cada projeto `v`
  vizinho de `u` ainda **não visitado nesta rodada** (`visitado[v] != rodada`), marque-o; se
  `parR[v] == -1` ou `aumenta(parR[v])` der certo, faça `parL[u] = v`, `parR[v] = u` e
  devolva 1. Se nenhum vizinho servir, devolva 0;
- `emparelhamento_maximo()` começa com tudo livre (`-1`) e, para cada aluno `u`, incrementa
  `rodada` e chama `aumenta(u)`. Devolve o número de sucessos.

O custo é $O(nm)$: no máximo $|L|$ buscas, cada uma $O(n + m)$. A entrada é `nL nR m` seguido
de $m$ arestas `u v` (aluno $u \in 0..nL-1$, projeto $v \in 0..nR-1$).

In [ ]:
%%writefile src/emparelhamento.c
/* Exercicio 4: emparelhamento bipartido maximo por caminhos alternantes (Kuhn): O(n m).
   Entrada: "nL nR m" e m linhas "u v" (aluno u em 0..nL-1, projeto v em 0..nR-1).
   TODO: complete aumenta() e emparelhamento_maximo(). */
#include <stdio.h>
#include <stdlib.h>

int nL, nR, m;
int *primeiro, *prox, *dest;    /* arestas do aluno u: a = primeiro[u], prox[a], ...; projeto dest[a] */
int *parL, *parR;               /* parL[u] = projeto de u, parR[v] = aluno de v (-1: livre) */
int *visitado, rodada;          /* visitado[v] == rodada: projeto v ja visto nesta busca */

/* procura caminho alternante a partir do aluno u; se achar, inverte-o e devolve 1 */
int aumenta( int u) {
   /* TODO */
   return 0;
}

/* devolve o tamanho do emparelhamento maximo (e deixa parL[] e parR[] preenchidos) */
int emparelhamento_maximo( void) {
   /* TODO */
   return 0;
}

/* ---------- leitura e saida (nao altere) ---------- */
int main( void) {
   if (scanf( "%d %d %d", &nL, &nR, &m) != 3) return 1;
   primeiro = malloc( nL * sizeof (int));
   prox = malloc( m * sizeof (int));  dest = malloc( m * sizeof (int));
   parL = malloc( nL * sizeof (int));  parR = malloc( nR * sizeof (int));
   visitado = calloc( nR, sizeof (int));
   for (int u = 0; u < nL; ++u) primeiro[u] = -1;
   int *us = malloc( m * sizeof (int)), *vs = malloc( m * sizeof (int));
   for (int i = 0; i < m; ++i)
      if (scanf( "%d %d", &us[i], &vs[i]) != 2) return 1;
   for (int i = m - 1; i >= 0; --i) {        /* insere ao contrario: lista na ordem da entrada */
      dest[i] = vs[i];  prox[i] = primeiro[us[i]];  primeiro[us[i]] = i;
   }
   int k = emparelhamento_maximo();
   printf( "tamanho=%d\n", k);
   for (int u = 0; u < nL; ++u)
      if (parL[u] != -1) printf( "%d %d\n", u, parL[u]);
   return 0;
}

In [ ]:
# Teste automático do Exercício 4

def emparelhamento_c(nL, nR, arestas, mostrar=False):
    entrada = f'{nL} {nR} {len(arestas)}\n' + ''.join(f'{u} {v}\n' for u, v in arestas)
    saida = rodar('emparelhamento.c', entrada=entrada, mostrar=mostrar).split('\n')
    pares = [tuple(map(int, l.split())) for l in saida[1:] if l.strip()]
    return campo(saida[0], 'tamanho'), pares

def emparelhamento_por_fluxo(nL, nR, arestas):
    """Tamanho do emparelhamento máximo pela rede s -> L -> R -> t com capacidades 1."""
    n = nL + nR + 2
    arcos = [(0, 1 + u, 1) for u in range(nL)] + [(1 + u, 1 + nL + v, 1) for u, v in arestas] \
          + [(1 + nL + v, n - 1, 1) for v in range(nR)]
    return fluxo_max_ref(n, arcos)[0]

# Exemplo dos slides: alunos A, B, C, D = 0..3; projetos 1..4 = 0..3.
ALUNOS = [(0, 0), (0, 1), (1, 0), (2, 1), (2, 2), (3, 2), (3, 3)]
k, pares = emparelhamento_c(4, 4, ALUNOS, mostrar=True)
assert k == 4 and len(pares) == 4, 'Verifique sua implementação (o exemplo dos slides tem emparelhamento perfeito)'

rng = random.Random(11)
for _ in range(300):
    nL, nR = rng.randint(1, 8), rng.randint(1, 8)
    arestas = sorted({(rng.randrange(nL), rng.randrange(nR)) for _ in range(rng.randint(0, 20))})
    k, pares = emparelhamento_c(nL, nR, arestas)
    assert all(p in arestas for p in pares), f'par fora do grafo em {arestas}'
    assert len({u for u, _ in pares}) == len({v for _, v in pares}) == len(pares) == k, f'não é emparelhamento: {pares}'
    assert k == emparelhamento_por_fluxo(nL, nR, arestas), f'não é máximo em {arestas}'
print('Exercício 4 ok!')

Note como o aluno $B$ (vértice 1) é encaixado no exemplo: quando chega a vez dele, o projeto 1
já é de $A$; a recursão desloca $A$ para o projeto 2, que desloca $C$ para o 3, que desloca $D$
para o 4. É exatamente o caminho alternante $B \to 1 \Rightarrow A \to 2 \Rightarrow C \to 3
\Rightarrow D \to 4$ dos slides.

## Desafio Final: ainda dá para ser campeão?

Faltando poucas rodadas para o fim de uma liga universitária (fictícia), a tabela é a seguinte.
Cada jogo dá uma vitória a um dos dois times (não há empate), e é campeão quem terminar com mais
vitórias (empates no topo contam como "ainda pode ser campeão").

| time | vitórias | faltam contra Ara | Ita | Lag | Est | Pro |
|---|---|---|---|---|---|---|
| Aracaju (Ara) | 75 | — | 12 | 2 | 6 | 3 |
| Itabaiana (Ita) | 71 | 12 | — | 2 | 6 | 4 |
| Lagarto (Lag) | 69 | 2 | 2 | — | 1 | 0 |
| Estância (Est) | 64 | 6 | 6 | 1 | — | 0 |
| Propriá (Pro) | 49 | 3 | 4 | 0 | 0 | — |

Um time $x$ está **trivialmente eliminado** se, mesmo vencendo todos os seus jogos, não alcança
quem já tem mais vitórias. Mas um time pode estar eliminado sem que isso seja óbvio: as
vitórias que faltam distribuir entre os **outros** times podem forçar algum deles a passar de
$x$.

**Tarefa.**

1. Suponha que $x$ vence todos os seus jogos e termina com $M$ vitórias. Modele como uma rede
   de fluxo a pergunta "dá para distribuir os resultados dos jogos *entre os outros times* sem que
   nenhum deles passe de $M$?". (Dica: um vértice para cada **par** de outros times que ainda se
   enfrentam, um vértice para cada outro time. Que capacidades põem na rede o número de jogos e o
   limite $M - w_i$?)
2. Use o seu Ford-Fulkerson (Exercício 2 ou `fluxo_max_ref`) para decidir quais times estão
   eliminados. Qual é eliminado sem que seja trivial?
3. Para o time eliminado de forma não trivial, use o **corte mínimo** (Exercício 3) para achar
   um conjunto $Q$ de times tal que $\frac{w(Q) + g(Q)}{|Q|} > M$, em que $w(Q)$ é o total de
   vitórias de $Q$ e $g(Q)$ o número de jogos restantes entre eles. Esse $Q$ é um certificado
   que qualquer torcedor consegue conferir sem computador.

In [ ]:
# Desafio: eliminação no campeonato
# TODO: resolva o desafio abaixo

TIMES = ['Aracaju', 'Itabaiana', 'Lagarto', 'Estância', 'Propriá']
VITORIAS = [75, 71, 69, 64, 49]
JOGOS = [[0, 12, 2, 6, 3],
         [12, 0, 2, 6, 4],
         [2, 2, 0, 1, 0],
         [6, 6, 1, 0, 0],
         [3, 4, 0, 0, 0]]

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. Cap. 10 (§10.2 fluxo máximo, §10.3 emparelhamento bipartido).
- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. Cap. 26 (§26.1 Figura 26.1; §26.2 Ford-Fulkerson, Teorema 26.6, Figura 26.7, Edmonds-Karp; §26.3 emparelhamento). (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Addison-Wesley, 2006. §7.1–7.3, §7.5 e §7.12 (eliminação no campeonato).
- DASGUPTA, S.; PAPADIMITRIOU, C.; VAZIRANI, U. *Algorithms*. McGraw-Hill, 2008. §7.2–7.3.
- FORD, L. R.; FULKERSON, D. R. Maximal Flow Through a Network. *Canadian Journal of Mathematics*, v. 8, p. 399–404, 1956.
- EDMONDS, J.; KARP, R. M. Theoretical Improvements in Algorithmic Efficiency for Network Flow Problems. *Journal of the ACM*, v. 19, n. 2, p. 248–264, 1972.